# pose17_v1 数据处理与可视化入口

这个 notebook 做两件事：

1. 从 `data/raw/` 读取安卓导出的原始 JSON，清洗并归一化成单样本 CSV。
2. 读取一个已经清洗好的 CSV，并把训练实际使用的骨架画出来。

通常你只需要修改下面配置 cell 里的三个变量。

## 1. 配置区：这里是主要需要你手动修改的地方

只改这三个变量：

- `RAW_JSON_FILENAME`：填写 `data/raw/` 里面的原始 JSON 文件名。
- `OUTPUT_LABEL_FOLDER`：填写处理后的 CSV 要放进哪个动作文件夹，例如 `1_lift_sky`。
- `VISUALIZE_CSV_PATH`：填写要可视化的 CSV 完整路径。

注意：`RAW_JSON_FILENAME` 只写文件名，不写完整路径；`VISUALIZE_CSV_PATH` 可以写完整 Windows 路径。

In [ ]:
RAW_JSON_FILENAME = "0004_firstaction1_20260408_202501.json"
OUTPUT_LABEL_FOLDER = "1_lift_sky"
VISUALIZE_CSV_PATH = r"D:\nottingham\research\Android_CV\android_app\data\1_lift_sky\0004_firstaction1_20260408_202501_normalized.csv"

VISIBILITY_THRESHOLD = 0.5
CSV_ROTATION = "counterclockwise_90"  # none, clockwise_90, counterclockwise_90, rotate_180
FRAME_STEP = 1

## 2. 初始化项目路径和导入函数

这个 cell 会自动定位 `android_app` 项目根目录，并把 `code/` 加入 Python 搜索路径。正常情况下不需要修改。

In [ ]:
from pathlib import Path
import importlib
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "code" else NOTEBOOK_DIR
CODE_DIR = PROJECT_ROOT / "code"
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import data_transform
import pose17_viz
import pose17_viz.io
import pose17_viz.plotting
import pose17_viz.transform

importlib.reload(data_transform)
importlib.reload(pose17_viz.io)
importlib.reload(pose17_viz.transform)
importlib.reload(pose17_viz.plotting)
importlib.reload(pose17_viz)

from data_transform import load_pose17_sample, process_one_json
from pose17_viz import (
    animate_normalized_csv,
    load_normalized_csv,
    preview_normalized_csv,
    summarize_normalized_csv,
)

raw_json_path = RAW_DIR / RAW_JSON_FILENAME
output_folder = DATA_DIR / OUTPUT_LABEL_FOLDER
visualize_csv_path = Path(VISUALIZE_CSV_PATH)

{
    "project_root": str(PROJECT_ROOT),
    "raw_json_path": str(raw_json_path),
    "output_folder": str(output_folder),
    "visualize_csv_path": str(visualize_csv_path),
}

## 3. 检查原始 JSON 是否能读取

这个 cell 会读取 `data/raw/<RAW_JSON_FILENAME>`，确认它是 `pose17_v1` 格式，并显示基本信息。这里不会生成 CSV。

In [ ]:
sample = load_pose17_sample(raw_json_path)

{
    "sample_id": sample.get("sample_id"),
    "subject_id": sample.get("subject_id"),
    "action_name": sample.get("action_name"),
    "schema": sample.get("landmark_schema_version"),
    "raw_frame_count": len(sample.get("frames", [])),
}

## 4. 清洗并导出单样本 CSV

这个 cell 会执行真正的数据处理：

- 跳过无 pose 帧。
- 跳过低可见度帧。
- 对 `x/y/z` 做髋中心平移归一化和躯干尺度缩放。
- 输出到 `data/<OUTPUT_LABEL_FOLDER>/<sample_id>_normalized.csv`。

In [ ]:
output_csv_path = process_one_json(
    input_json=raw_json_path,
    output_dir=output_folder,
    label=OUTPUT_LABEL_FOLDER,
    threshold=VISIBILITY_THRESHOLD,
)

output_csv_path

## 5. 读取要可视化的 CSV 并查看摘要

这里读取的是 `VISUALIZE_CSV_PATH` 指定的 CSV。这样可视化和刚才的清洗步骤是解耦的：你可以可视化刚生成的 CSV，也可以填入任何已有 CSV 的完整路径。

In [ ]:
cleaned = load_normalized_csv(visualize_csv_path)
summary = summarize_normalized_csv(cleaned)
summary

## 6. 静态预览清洗后的骨架

这个 cell 会显示首帧、中间帧和末帧。它读取的是清洗后的 CSV，不是原始 JSON。

In [ ]:
preview_normalized_csv(cleaned, rotation=CSV_ROTATION)

## 7. 动画预览清洗后的骨架

这个 cell 会把 CSV 里的帧按时间顺序播放出来，用来检查训练实际看到的动作序列是否合理。`FRAME_STEP` 可以调大，比如改成 `2` 或 `3`，这样播放会更轻。

In [ ]:
animate_normalized_csv(
    cleaned,
    rotation=CSV_ROTATION,
    interval_ms=33,
    frame_step=FRAME_STEP,
)